# Reditus — Segmentation des patients et prédiction de la réadmission à 30 jours chez les patients diabétiques
*Lab de préparation des données, clustering et classification — dataset Diabetes 130-US Hospitals*

**Dataset :** 101 766 séjours hospitaliers de patients diabétiques (1999–2008) + `IDS_mapping.csv` (dictionnaire pour décoder les identifiants d'admission et de sortie).

**Plan du notebook :** 0. État de l'art → 1. Objectifs (BO / DSO) → 2. Chargement → 3. Exploration et visualisation → 4. Nettoyage et transformation (chaque choix justifié) → 5. Clustering → 6. Classification → 7. Conclusions.

## 0. État de l'art

*Objectif de cette section : situer le travail par rapport à ce qui existe déjà, pour justifier nos choix (modèles, métriques) et disposer d'un repère réaliste de performance.*

**Origine des données.** Le dataset est un extrait de 10 ans (1999–2008) de soins cliniques dans 130 hôpitaux américains (base Health Facts). Il a été analysé par Strack et al. (2014), qui ont étudié le lien entre la **mesure de l'HbA1c** à l'hôpital et la réadmission précoce, à l'aide d'une **régression logistique multivariée**. Ils rapportent que l'HbA1c n'était mesurée que dans 18,4 % des séjours et que son lien avec la réadmission dépend du diagnostic principal. Le dataset a ensuite été déposé sur le dépôt UCI Machine Learning Repository.

**Pratiques courantes sur ce type de problème** *(synthèse générale, à vérifier avec des sources avant citation)*
* **Modèles :** régression logistique (référence interprétable), Random Forest et gradient boosting ; l'apprentissage profond apporte en général peu sur ce type de données tabulaires.
* **Performance :** la prédiction de la réadmission est réputée difficile ; les AUC rapportés se situent le plus souvent autour de 0,6 à 0,7, y compris pour des scores cliniques comme l'indice LACE, car des facteurs décisifs (soutien social, observance, soins après la sortie) ne figurent pas dans les données.
* **Contexte réglementaire :** aux États-Unis, le programme Medicare de réduction des réadmissions (HRRP) pénalise les hôpitaux pour des réadmissions excessives dans certaines pathologies, ce qui explique l'intérêt financier du sujet.

**Pièges fréquents** : patients répétés qui provoquent une fuite entre train et test, conservation des patients décédés ou en hospice (non réadmissibles), usage de l'accuracy malgré le déséquilibre des classes, et regroupement insuffisant des codes diagnostics.

**Positionnement de Reditus**
* On évite les trois premiers pièges (un séjour par patient, exclusion décès/hospice, métriques adaptées au déséquilibre).
* On **combine segmentation et prédiction** : la segmentation explique *qui* sont les patients, la classification prédit *qui reviendra*, ce que les études centrées sur un seul modèle de prédiction font rarement.
* Repère de performance attendu : une AUC proche de 0,65 serait cohérente avec les ordres de grandeur de la littérature.

**Références**
* Strack B., DeShazo J. P., Gennings C., Olmo J. L., Ventura S., Cios K. J., Clore J. N. (2014). *Impact of HbA1c Measurement on Hospital Readmission Rates: Analysis of 70,000 Clinical Database Patient Records.* BioMed Research International, 2014, 781670. doi:10.1155/2014/781670
* Strack B. et al. (2014). *Diabetes 130-US Hospitals for Years 1999–2008* [dataset]. UCI Machine Learning Repository.
* À ajouter après vérification : van Walraven C. et al. (2010), indice LACE, *CMAJ* ; documentation du programme Medicare HRRP.

## 1. De l'objectif métier (BO) à l'objectif Data Science (DSO)

| | Objectif métier (BO) | Objectif Data Science (DSO) |
|---|---|---|
| **Classification** | Réduire les réadmissions évitables à 30 jours (coûts + pénalités Medicare + préjudice pour le patient) en ciblant le suivi sur les patients à haut risque | Prédire `readmitted == '<30'` à la sortie. Succès = bon **recall / PR-AUC** sur la classe minoritaire (~11 % de positifs, ~9 % une fois un seul séjour gardé par patient) |
| **Clustering** | Personnaliser les parcours de soins selon le type de patient | Trouver 3 à 6 segments de patients interprétables (silhouette, Davies-Bouldin) et comparer leur risque de réadmission |

### 1.1 Compréhension métier — BO1 : Réduire les réadmissions évitables à 30 jours

| Élément | Description |
|---|---|
| **Contexte** | Le diabète est une maladie chronique aux complications fréquentes. Les hôpitaux réadmettent régulièrement des patients diabétiques peu après leur sortie. |
| **Problème métier** | Une réadmission sous 30 jours signale souvent une guérison incomplète, une sortie trop précoce ou un suivi insuffisant. Elle est coûteuse pour l'hôpital (lits, personnel, pénalités financières dans de nombreux systèmes d'assurance santé comme le programme de réadmission de Medicare) et néfaste pour le patient. |
| **Parties prenantes** | Direction de l'hôpital (coûts, indicateurs qualité), médecins et infirmiers (décisions de soin), coordinateurs de soins (suivi), patients. |
| **Objectif métier** | Identifier **à la sortie** les patients les plus susceptibles de revenir sous 30 jours, afin de concentrer les ressources de suivi limitées (appel, rendez-vous précoce, revue des médicaments) sur ceux qui en ont le plus besoin. |
| **Critères de succès métier** | Moins de réadmissions à 30 jours ; effort de suivi concentré sur une petite part des patients (par ex. les 20 % de scores de risque les plus élevés capturent une grande part des réadmissions). |
| **Tâche Data Science** | Classification binaire de `readmitted == '<30'`. |
| **Contraintes et risques** | Fort déséquilibre des classes (~9–11 % de positifs) ; manquer un patient à haut risque coûte plus cher qu'un appel inutile ; les prédictions sont une aide au tri, pas une décision clinique ; l'équité selon l'âge, l'origine et le genre doit être surveillée. |

### 1.2 Compréhension métier — BO2 : Personnaliser les soins grâce à la segmentation des patients

| Élément | Description |
|---|---|
| **Contexte** | Les patients diabétiques hospitalisés ne forment pas une population homogène : un jeune patient en séjour programmé court et un patient âgé avec de nombreuses comorbidités et des admissions répétées n'ont pas les mêmes besoins. |
| **Problème métier** | Un protocole de sortie et de suivi identique pour tous gaspille des ressources sur les patients à faible risque et sert mal les patients complexes. L'hôpital n'a pas de typologie claire de ses patients diabétiques. |
| **Parties prenantes** | Responsables de parcours de soins, chefs de service, planificateurs de l'hôpital et direction financière (allocation des ressources). |
| **Objectif métier** | Construire un petit nombre de **profils de patients interprétables** et concevoir un parcours de soins adapté à chacun (par ex. suivi pharmaceutique pour les patients dont le traitement change souvent, gestion de cas pour les usagers fréquents des urgences). |
| **Critères de succès métier** | 3 à 6 segments compréhensibles par les cliniciens, assez grands pour être actionnables, et différents par leurs besoins et leur risque de réadmission. |
| **Tâche Data Science** | Clustering non supervisé (K-Means) sur des variables cliniques et d'utilisation des soins, puis profilage de chaque segment, y compris son taux de réadmission (utilisé uniquement *après* le clustering, pas pour le construire). |
| **Contraintes et risques** | Pas de vérité terrain : la qualité se juge par la silhouette / Davies-Bouldin et par l'interprétabilité clinique ; les segments peuvent se chevaucher ; le résultat dépend des variables choisies et de la mise à l'échelle. |

**Lien entre les deux objectifs :** le BO2 explique *qui* sont les patients, le BO1 prédit *qui reviendra*. Ensemble, ils permettent d'adapter le parcours par segment et de prioriser, au sein de chaque segment, selon le score de risque.